In [30]:
import pandas as pd 
import joblib
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import (
    precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
)
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

# Modèles
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

df= pd.read_csv('../data/cleaned.csv')
prep = joblib.load('../models/clustering/preprocessor.pkl')
kmeans = joblib.load('../models/clustering/kmeans_churn_model.pkl')

In [31]:
x=df.drop('Churn',axis=1)
y=df['Churn']

In [32]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y=le.fit_transform(y)
y

array([0, 0, 1, ..., 0, 1, 0], shape=(7043,))

In [33]:
from sklearn.model_selection import train_test_split

x_train,x_test,y_train,y_test= train_test_split(
    x,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [34]:
x_train_prep = prep.transform(x_train)
x_test_prep = prep.transform(x_test)

x_train_scaled = pd.DataFrame(x_train_prep, index=x_train.index)
x_test_scaled = pd.DataFrame(x_test_prep, index=x_test.index)

x_train_scaled['cluster'] = kmeans.predict(x_train_prep)
x_test_scaled['cluster'] = kmeans.predict(x_test_prep)

x_train_scaled.columns = x_train_scaled.columns.astype(str)
x_test_scaled.columns = x_test_scaled.columns.astype(str)

In [35]:


models = {
    'Logistic Regression': {
        'model': LogisticRegression(random_state=42, max_iter=1000),
        'params': {
            'model__C': [0.1, 1.0, 10.0],
            'model__solver': ['lbfgs', 'liblinear']
        }
    },
    'Decision Tree': {
        'model': DecisionTreeClassifier(random_state=42),
        'params': {
            'model__max_depth': [3, 5, 10, None],
            'model__min_samples_split': [2, 5, 10]
        }
    },
    'Random Forest': {
        'model': RandomForestClassifier(random_state=42),
        'params': {
            'model__n_estimators': [50, 100],
            'model__max_depth': [5, 10, None]
        }
    },
    'SVM': {
        'model': SVC(random_state=42, probability=True),
        'params': {
            'model__C': [0.1, 1, 10],
            'model__kernel': ['rbf', 'linear']
        }
    },
    'Gradient Boosting': {
        'model': GradientBoostingClassifier(random_state=42),
        'params': {
            'model__n_estimators': [50, 100],
            'model__learning_rate': [0.01, 0.1, 0.2]
        }
    },
    'XGBoost': {
        'model': XGBClassifier(random_state=42, eval_metric='logloss'),
        'params': {
            'model__n_estimators': [50, 100],
            'model__max_depth': [3, 6],
            'model__learning_rate': [0.01, 0.1]
        }
    }
}

In [36]:
res=[]
best_e={}
pred={}

for name, config in models.items():
    print(f"--- Entraînement : {name} ---")
    
    pipeline = Pipeline([
        ('smote', SMOTE(random_state=42)),
        ('model', config['model'])
    ])
    
    grid = GridSearchCV(
        estimator=pipeline,
        param_grid=config['params'],
        scoring='f1',
        cv=5,
        n_jobs=-1
    )
    
    grid.fit(x_train_scaled, y_train)
    
    best_model = grid.best_estimator_
    best_e[name] = best_model
    
    y_pred = best_model.predict(x_test_scaled)
    y_proba = best_model.predict_proba(x_test_scaled)[:, 1]
    pred[name] = y_pred
    
    res.append({
        'Modèle': name,
        'Best Params': grid.best_params_,
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1-Score': f1_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, y_proba)
    })

--- Entraînement : Logistic Regression ---
--- Entraînement : Decision Tree ---
--- Entraînement : Random Forest ---
--- Entraînement : SVM ---


/home/yassine/Projets/workers/.venv/lib/python3.14/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/yassine/Projets/workers/.venv/lib/python3.14/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/yassine/Projets/workers/.venv/lib/python3.14/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/yassine/Projets/workers/.venv/lib/python3.14/site-packages/sklearn/svm/_base.py:236: FutureWarning: The

--- Entraînement : Gradient Boosting ---
--- Entraînement : XGBoost ---


In [37]:
df_results = pd.DataFrame(res).sort_values(by='F1-Score', ascending=False)
display(df_results[['Modèle', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']])

,Modèle,Precision,Recall,F1-Score,ROC-AUC
2,Random Forest,0.527473,0.770053,0.626087,0.842571
4,Gradient Boosting,0.543825,0.729947,0.623288,0.841248
0,Logistic Regression,0.507666,0.796791,0.620187,0.840463
3,SVM,0.506108,0.775401,0.612460,0.835457
1,Decision Tree,0.549356,0.684492,0.609524,0.821641
5,XGBoost,0.553846,0.673797,0.607961,0.842453
